# Module 0: PyTorch 前置知识

## 学习目标
在学习 Mini-SGLang 之前，你需要掌握以下 PyTorch 基础知识：

1. **Tensor 基础** - 创建、操作、形状变换
2. **CUDA 基础** - GPU 设备管理、同步
3. **CUDA Stream** - 异步执行与流同步
4. **CUDA Event** - 性能计时
5. **CUDA Graph** - 减少 CPU 开销
6. **nn.Module** - 模型结构
7. **分布式通信** - torch.distributed 基础
8. **内存管理** - Pinned Memory, Contiguous

---

## 0.1 Tensor 基础

Mini-SGLang 大量使用 Tensor 操作，以下是最常用的：

In [ ]:
import torch
import torch.nn as nn

# ========== 1. 创建 Tensor ==========

# 指定数据类型 - Mini-SGLang 常用 int32 存储 token IDs
input_ids = torch.tensor([1, 2, 3, 4, 5], dtype=torch.int32)
print(f"input_ids: {input_ids}, dtype: {input_ids.dtype}")

# 创建空 tensor (不初始化，节省时间)
# 在 KV Cache 预分配时常用
empty_cache = torch.empty(32, 128, dtype=torch.bfloat16)
print(f"empty_cache shape: {empty_cache.shape}")

# 创建全零 tensor
page_table = torch.zeros(100, dtype=torch.int32)
print(f"page_table: {page_table[:5]}")

# 创建指定形状的 tensor
hidden_states = torch.randn(1, 512, 1024)  # (batch, seq_len, hidden_size)
print(f"hidden_states shape: {hidden_states.shape}")

In [ ]:
# ========== 2. Tensor 拼接 (torch.cat) ==========
# 在 KV Cache 追加新 token 时常用

# 沿着 dim=0 拼接 (最常用)
t1 = torch.tensor([1, 2, 3])
t2 = torch.tensor([4, 5])
combined = torch.cat([t1, t2])
print(f"cat along dim=0: {combined}")

# 沿着 dim=1 拼接 (在 attention 中使用)
k_cache = torch.randn(1, 10, 128)  # (batch, seq, head_dim)
k_new = torch.randn(1, 1, 128)     # 新 token 的 K
k_updated = torch.cat([k_cache, k_new], dim=1)
print(f"K cache: {k_cache.shape} -> {k_updated.shape}")

In [ ]:
# ========== 3. 形状变换 ==========

# view: 改变形状 (必须连续)
x = torch.randn(2, 3, 4)
x_flat = x.view(-1)  # 展平为 1D
print(f"view: {x.shape} -> {x_flat.shape}")

# reshape: 更灵活的形状变换
x_reshaped = x.reshape(6, 4)
print(f"reshape: {x.shape} -> {x_reshaped.shape}")

# transpose: 交换维度 (attention 中常用)
# (batch, seq, num_heads, head_dim) -> (batch, num_heads, seq, head_dim)
x = torch.randn(2, 10, 8, 64)  # batch=2, seq=10, heads=8, dim=64
x_transposed = x.transpose(1, 2)
print(f"transpose(1,2): {x.shape} -> {x_transposed.shape}")

# unsqueeze/squeeze: 添加/移除维度
x = torch.randn(10, 64)
x_unsqueezed = x.unsqueeze(0)  # 在位置 0 添加维度
print(f"unsqueeze(0): {x.shape} -> {x_unsqueezed.shape}")

In [ ]:
# ========== 4. 索引与切片 ==========

# 基本索引
x = torch.arange(10)
print(f"x[3]: {x[3]}")
print(f"x[2:5]: {x[2:5]}")
print(f"x[-3:]: {x[-3:]}")

# 高级索引 (用索引 tensor 选择元素)
# 这在 Page Table 查找时非常重要
indices = torch.tensor([1, 3, 5, 7])
selected = x[indices]
print(f"x[{indices.tolist()}]: {selected}")

# 布尔索引
mask = x > 5
print(f"x[x > 5]: {x[mask]}")

In [ ]:
# ========== 5. 原地操作 (节省内存) ==========

# copy_: 原地复制数据
src = torch.tensor([1.0, 2.0, 3.0])
dst = torch.empty(3)
dst.copy_(src)  # dst = src，但不创建新 tensor
print(f"copy_: dst = {dst}")

# fill_: 原地填充
x = torch.empty(5)
x.fill_(0.5)
print(f"fill_: {x}")

# 在 Mini-SGLang 中，原地操作用于更新 KV Cache
# 例如: kv_cache[indices].copy_(new_kv)

## 0.2 CUDA 基础

Mini-SGLang 是 GPU 推理框架，需要了解 CUDA 操作。

In [ ]:
# ========== 1. 检查 CUDA 可用性 ==========
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")

In [ ]:
# ========== 2. 设备管理 ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 创建 GPU tensor
x_gpu = torch.randn(100, 100, device=device)
print(f"x_gpu device: {x_gpu.device}")

# 移动 tensor 到 GPU
x_cpu = torch.randn(100, 100)
x_moved = x_cpu.to(device)
print(f"Moved tensor device: {x_moved.device}")

# 移动模型到 GPU
model = nn.Linear(100, 100).to(device)
print(f"Model device: {next(model.parameters()).device}")

In [ ]:
# ========== 3. CUDA 同步 ==========
# GPU 操作是异步的，有时需要等待完成

if torch.cuda.is_available():
    import time
    
    x = torch.randn(1000, 1000, device="cuda")
    
    # 不同步：CPU 不等待 GPU 完成
    start = time.time()
    for _ in range(100):
        y = torch.matmul(x, x)
    end = time.time()
    print(f"Without sync: {(end-start)*1000:.2f} ms (not accurate!)")
    
    # 同步：等待 GPU 完成所有操作
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        y = torch.matmul(x, x)
    torch.cuda.synchronize()
    end = time.time()
    print(f"With sync: {(end-start)*1000:.2f} ms (accurate)")

## 0.3 CUDA Stream (关键概念!)

**Stream** 是 GPU 上的命令队列。同一个 Stream 内的操作按顺序执行，不同 Stream 可以并行执行。

Mini-SGLang 使用多个 Stream 实现 **Overlap Scheduling**：
- Scheduler Stream: 调度下一批任务
- Engine Stream: 执行模型前向传播

```
源代码参考: python/minisgl/scheduler/scheduler.py:23-30
```

In [ ]:
if torch.cuda.is_available():
    # ========== 1. 创建和使用 Stream ==========
    
    # 默认 Stream
    default_stream = torch.cuda.current_stream()
    print(f"Default stream: {default_stream}")
    
    # 创建新 Stream
    my_stream = torch.cuda.Stream()
    print(f"My stream: {my_stream}")
    
    # 在指定 Stream 上执行
    x = torch.randn(1000, 1000, device="cuda")
    with torch.cuda.stream(my_stream):
        y = torch.matmul(x, x)  # 在 my_stream 上执行
    
    print("Operations submitted to stream")

In [ ]:
if torch.cuda.is_available():
    # ========== 2. Stream 同步 ==========
    
    stream_a = torch.cuda.Stream()
    stream_b = torch.cuda.Stream()
    
    x = torch.randn(1000, 1000, device="cuda")
    
    # Stream A 上执行
    with torch.cuda.stream(stream_a):
        y = torch.matmul(x, x)
    
    # Stream B 需要等待 Stream A 的结果
    # 关键：stream_b.wait_stream(stream_a)
    stream_b.wait_stream(stream_a)
    
    with torch.cuda.stream(stream_b):
        z = torch.matmul(y, y)  # 安全使用 y 的结果
    
    # 同步所有操作
    torch.cuda.synchronize()
    print(f"Result shape: {z.shape}")
    
    # Mini-SGLang 中的用法:
    # scheduler/scheduler.py:100
    # self.engine.stream.wait_stream(self.stream)

## 0.4 CUDA Event (性能计时)

Event 用于在 GPU Stream 中记录时间点，精确测量 GPU 操作耗时。

```
源代码参考: python/minisgl/benchmark/perf.py:20-21
```

In [ ]:
if torch.cuda.is_available():
    # 创建计时 Event
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    x = torch.randn(2000, 2000, device="cuda")
    
    # 记录开始时间
    start_event.record()
    
    # 执行操作
    for _ in range(100):
        y = torch.matmul(x, x)
    
    # 记录结束时间
    end_event.record()
    
    # 等待完成并计算时间
    torch.cuda.synchronize()
    elapsed_time = start_event.elapsed_time(end_event)  # 毫秒
    
    print(f"100 matmul operations: {elapsed_time:.2f} ms")
    print(f"Average per operation: {elapsed_time/100:.3f} ms")

## 0.5 CUDA Graph (重要优化!)

**问题**: 在 Decode 阶段，每个 step 只处理少量 token，GPU 计算很快完成，但 CPU 调度 kernel 的开销相对较大。

**解决方案**: CUDA Graph 将一系列 GPU 操作"录制"成一个图，之后可以整体"回放"，减少 CPU 开销。

```
源代码参考: python/minisgl/engine/graph.py
```

In [ ]:
if torch.cuda.is_available():
    import time
    
    # 准备模型和输入
    model = nn.Sequential(
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
    ).cuda()
    
    # 静态输入 (CUDA Graph 要求输入形状固定)
    static_input = torch.randn(1, 256, device="cuda")
    
    # ========== 1. 不使用 CUDA Graph ==========
    # 预热
    for _ in range(10):
        _ = model(static_input)
    torch.cuda.synchronize()
    
    # 测量
    start = time.time()
    for _ in range(1000):
        output = model(static_input)
    torch.cuda.synchronize()
    normal_time = time.time() - start
    print(f"Without CUDA Graph: {normal_time*1000:.2f} ms for 1000 iterations")

In [ ]:
if torch.cuda.is_available():
    # ========== 2. 使用 CUDA Graph ==========
    
    # Step 1: 预热 (确保 CUDA 初始化完成)
    for _ in range(3):
        _ = model(static_input)
    torch.cuda.synchronize()
    
    # Step 2: 录制 Graph
    g = torch.cuda.CUDAGraph()
    
    # 在 capture 模式下录制
    with torch.cuda.graph(g):
        static_output = model(static_input)
    
    print(f"Graph captured!")
    print(f"  Input address: {static_input.data_ptr()}")
    print(f"  Output address: {static_output.data_ptr()}")
    
    # Step 3: 回放 Graph
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(1000):
        # 更新输入 (直接写入静态 buffer)
        static_input.copy_(torch.randn(1, 256, device="cuda"))
        # 回放 Graph
        g.replay()
    torch.cuda.synchronize()
    graph_time = time.time() - start
    
    print(f"\nWith CUDA Graph: {graph_time*1000:.2f} ms for 1000 iterations")
    print(f"Speedup: {normal_time/graph_time:.2f}x")

### CUDA Graph 注意事项

1. **静态形状**: 输入/输出形状必须固定
2. **静态地址**: 使用 `copy_()` 更新数据，不能重新分配内存
3. **无控制流**: Graph 内不能有 if/else, loops 等 Python 控制流
4. **Mini-SGLang 策略**: 为不同 batch size 预录制多个 Graph

```python
# 参考: engine/graph.py
class CudaGraphRunner:
    def __init__(self):
        self.graphs = {}  # batch_size -> graph
    
    def capture(self, batch_size: int, ...):
        # 为特定 batch_size 录制 graph
        ...
```

## 0.6 nn.Module 结构

Mini-SGLang 的模型继承自 PyTorch 的 `nn.Module`。

In [ ]:
# ========== 模型基本结构 ==========

class SimpleTransformerLayer(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()  # 必须调用父类构造函数
        
        # 子模块会自动注册
        self.attention = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size),
        )
        self.norm2 = nn.LayerNorm(hidden_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Attention with residual
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        
        # FFN with residual
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

# 创建模型
layer = SimpleTransformerLayer(hidden_size=256, num_heads=8)

# 查看参数
print("Model parameters:")
for name, param in layer.named_parameters():
    print(f"  {name}: {param.shape}")

In [ ]:
# ========== 模型加载权重 ==========

# Mini-SGLang 从 safetensors 加载预训练权重
# 参考: models/qwen3.py load_params 函数

# 模拟权重加载
dummy_weights = {
    'attention.in_proj_weight': torch.randn(768, 256),
    'attention.in_proj_bias': torch.randn(768),
    'attention.out_proj.weight': torch.randn(256, 256),
    'attention.out_proj.bias': torch.randn(256),
}

# 加载部分权重
missing, unexpected = layer.load_state_dict(dummy_weights, strict=False)
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

## 0.7 torch.distributed 基础

Tensor Parallelism 使用 `torch.distributed` 进行 GPU 间通信。

```
源代码参考: python/minisgl/distributed/impl.py
```

In [ ]:
# ========== 分布式概念 ==========
# 注意：这些代码在多进程环境下才能运行
# 这里只展示概念

import torch.distributed as dist

# 核心概念:
# - world_size: 总共有多少个进程 (GPU)
# - rank: 当前进程的编号 (0, 1, 2, ...)

# 初始化进程组 (通常在程序开始时调用一次)
# dist.init_process_group(backend="nccl")  # NVIDIA GPU 使用 NCCL

# 常用通信操作:
print("""
1. all_reduce: 所有 GPU 的 tensor 求和，结果广播给所有 GPU
   用途: Row Parallel 后合并结果
   
   GPU 0: [1, 2]  ─┐
   GPU 1: [3, 4]  ─┼─> all_reduce(sum) -> GPU 0: [4, 6]
                        GPU 1: [4, 6]

2. all_gather: 收集所有 GPU 的 tensor，拼接后广播
   用途: Column Parallel 后合并结果
   
   GPU 0: [a]     ─┐
   GPU 1: [b]     ─┼─> all_gather -> GPU 0: [a, b]
                          GPU 1: [a, b]
""")

In [ ]:
# Mini-SGLang 中的分布式实现
# 参考: distributed/impl.py

class MockDistributedEngine:
    """模拟 Mini-SGLang 的分布式引擎"""
    
    def __init__(self, world_size: int, rank: int):
        self.world_size = world_size
        self.rank = rank
    
    def all_reduce(self, x: torch.Tensor) -> torch.Tensor:
        """所有 GPU 求和"""
        # 实际实现:
        # dist.all_reduce(x, op=dist.ReduceOp.SUM)
        # return x
        return x  # 单 GPU 模拟
    
    def all_gather(self, x: torch.Tensor) -> torch.Tensor:
        """收集并拼接所有 GPU 的 tensor"""
        # 实际实现:
        # output = torch.empty(self.world_size * x.shape[0], ..., device=x.device)
        # dist.all_gather_into_tensor(output, x)
        # return output
        return x  # 单 GPU 模拟

# 使用示例
engine = MockDistributedEngine(world_size=2, rank=0)
x = torch.randn(4, 4)
result = engine.all_reduce(x)
print(f"all_reduce result shape: {result.shape}")

## 0.8 内存管理

高效的内存管理对推理性能至关重要。

In [ ]:
# ========== 1. 连续内存 (contiguous) ==========

x = torch.randn(3, 4)
print(f"Original is contiguous: {x.is_contiguous()}")

# transpose 后可能不连续
x_t = x.transpose(0, 1)
print(f"After transpose is contiguous: {x_t.is_contiguous()}")

# 调用 contiguous() 确保连续
x_contig = x_t.contiguous()
print(f"After .contiguous() is contiguous: {x_contig.is_contiguous()}")

# 为什么重要?
# - 很多 CUDA kernel 要求连续内存
# - FlashAttention 等高效算法需要连续的 KV Cache

In [ ]:
# ========== 2. Pinned Memory (页锁定内存) ==========
# CPU 和 GPU 之间高效传输数据

import time

size = (10000, 1000)

# 普通 CPU tensor
normal_tensor = torch.randn(size)

# Pinned memory tensor
pinned_tensor = torch.randn(size).pin_memory()

print(f"Normal tensor is pinned: {normal_tensor.is_pinned()}")
print(f"Pinned tensor is pinned: {pinned_tensor.is_pinned()}")

if torch.cuda.is_available():
    # 测量传输速度
    torch.cuda.synchronize()
    
    # 普通传输
    start = time.time()
    for _ in range(10):
        _ = normal_tensor.to("cuda", non_blocking=True)
    torch.cuda.synchronize()
    normal_time = time.time() - start
    
    # Pinned 传输 (异步，更高效)
    start = time.time()
    for _ in range(10):
        _ = pinned_tensor.to("cuda", non_blocking=True)
    torch.cuda.synchronize()
    pinned_time = time.time() - start
    
    print(f"\nNormal transfer: {normal_time*1000:.2f} ms")
    print(f"Pinned transfer: {pinned_time*1000:.2f} ms")
    print(f"Speedup: {normal_time/pinned_time:.2f}x")

In [ ]:
if torch.cuda.is_available():
    # ========== 3. GPU 内存查询 ==========
    
    # 当前已分配内存
    allocated = torch.cuda.memory_allocated() / 1024**3
    
    # 最大已分配内存
    max_allocated = torch.cuda.max_memory_allocated() / 1024**3
    
    # 缓存的内存 (PyTorch 内存池)
    cached = torch.cuda.memory_reserved() / 1024**3
    
    print(f"GPU Memory:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Max Allocated: {max_allocated:.2f} GB")
    print(f"  Reserved (Cached): {cached:.2f} GB")
    
    # 清理缓存
    torch.cuda.empty_cache()
    print(f"\nAfter empty_cache:")
    print(f"  Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## 0.9 数据类型 (dtype)

LLM 推理常用的数据类型及其内存占用。

In [ ]:
# ========== 常用数据类型 ==========

dtypes = {
    'float32 (FP32)': torch.float32,
    'float16 (FP16)': torch.float16,
    'bfloat16 (BF16)': torch.bfloat16,  # 推荐用于 LLM
    'int32': torch.int32,               # Token IDs, Page Table
    'int8': torch.int8,                 # 量化
}

print("Data type sizes:")
for name, dtype in dtypes.items():
    x = torch.empty(1, dtype=dtype)
    print(f"  {name}: {x.element_size()} bytes")

# 为什么用 bfloat16?
print("\n为什么 BF16 适合 LLM:")
print("  - FP16: 5 位指数, 10 位尾数 -> 更高精度，但范围小")
print("  - BF16: 8 位指数, 7 位尾数  -> 和 FP32 相同范围，精度够用")
print("  - LLM 训练常用 BF16，推理也应使用 BF16 以保持一致")

In [ ]:
# ========== dtype 转换 ==========

x_fp32 = torch.randn(100, 100)
x_bf16 = x_fp32.to(torch.bfloat16)

print(f"FP32 tensor size: {x_fp32.numel() * x_fp32.element_size()} bytes")
print(f"BF16 tensor size: {x_bf16.numel() * x_bf16.element_size()} bytes")
print(f"Memory savings: 50%")

# 精度损失检查
x_back = x_bf16.to(torch.float32)
error = (x_fp32 - x_back).abs().mean()
print(f"\nAverage conversion error: {error:.6f}")

## 0.10 推理模式

推理时禁用梯度计算，节省内存和计算。

In [ ]:
model = nn.Linear(256, 256)
x = torch.randn(1, 256)

# ========== 1. torch.no_grad() ==========
# 不计算梯度，节省内存
with torch.no_grad():
    y = model(x)
    print(f"With no_grad: requires_grad = {y.requires_grad}")

# ========== 2. torch.inference_mode() ==========
# 更激进的推理优化 (PyTorch 1.9+)
with torch.inference_mode():
    y = model(x)
    print(f"With inference_mode: requires_grad = {y.requires_grad}")

# ========== 3. model.eval() ==========
# 切换到评估模式 (影响 Dropout, BatchNorm 等)
model.eval()
print(f"Model training mode: {model.training}")

# Mini-SGLang 使用:
# @torch.inference_mode()
# def forward(...):
#     ...

## 小结

在学习 Mini-SGLang 之前，确保你理解了以下 PyTorch 概念:

| 概念 | 用途 | 重要性 |
|------|------|--------|
| Tensor 操作 | 数据处理基础 | ⭐⭐⭐⭐⭐ |
| CUDA Stream | Overlap Scheduling | ⭐⭐⭐⭐⭐ |
| CUDA Graph | Decode 加速 | ⭐⭐⭐⭐ |
| nn.Module | 模型结构 | ⭐⭐⭐⭐ |
| torch.distributed | Tensor Parallelism | ⭐⭐⭐ |
| dtype (bfloat16) | 内存优化 | ⭐⭐⭐ |
| Pinned Memory | CPU-GPU 传输 | ⭐⭐ |

---

**下一步**: [Module 1: LLM 推理基础](./01_llm_inference_basics.ipynb) - 理解 Prefill/Decode 和 KV Cache。